L'idea centrale dietro LSI è che parole e documenti che non condividono esattamente gli stessi termini, possono comunque essere vicini in uno spazio semantico latente. Ad esempio, se due documenti parlano entrambi di "automobili" ma uno usa la parola "auto" e l'altro "veicoli", LSI può riconoscere che questi termini sono correlati e posizionare i documenti vicini nello spazio semantico.

Si sfrutta la SVD Singular Value Decomposition per decomporre la matrice dei termini-documenti in tre matrici: 
$$ A = U \Sigma V^T $$
dove:
-$U$ è la matrice contenente come colonne i vettori latenti associati ai termini
- $\Sigma$ è una matrice diagonale che contiene i valori singolari, che rappresentano l'importanza dei concetti latenti, ossia rappresentano **quanta informazione è catturata da ciascuna dimensione latente** (in particolare sono la radice degli autovalori della matrice $A^TA$ o $AA^T$)
- $V$ è la matrice contenente i vettori latenti associati ai documenti

Possiamo in un certo senso misuare quanta informazione è spiegata mantenendo solo le prime $k$ dimensioni latenti (secondo la tecnica di low rank approximation) tramite la formula:
$$ \text{Informazione spiegata} = \frac{\sum_{i=1}^{k} \sigma_i^2}{\sum_{i=1}^{r} \sigma_i^2} $$
dove $\sigma_i$ sono i valori singolari ordinati in ordine decrescente e $r$ è il rango della matrice originale $A$. 

Spiegandolo in modo molto intuitivo (in realtà è ben più complesso) nella tabella sotto vediamo quindi che mantenendo la prima dimensione latente riusciamo a spiegare circa il 35% dell'informazione, con la seconda arriviamo al 62%, con la terza al 78% etc..  

<img src="img/tabella.png" alt="Tabella informazione spiegata" width="450">

Fare questo tipo di analisi è importante per capire a quale valore di $k$ è opportuno fermarsi, bilanciando la quantità di informazione mantenuta con la complessità del modello. Chiaramente qui parliamo di un toy example con solo 6 documenti, infatti con $k=6$ si spiega il 100% dell'informazione in quanto si torna alla matrice originale.  
In scenari reali tuttavia milioni di documenti, non possiamo permetterci di calcolare l'energia cumulativa per ogni possibile valore di $k$ ma possiamo usare tecniche da benchmark per stimare un buon valore.  
Perché approssimare? Per eliminare il rumore: si perdono i dettagli della iperdimensionalità originale ma così facendo possiamo catturare la struttura semantica principale.

Molto importante il fatto che approssimando $A$, si passa da una matrice sparsa a una matrice densa, non posso più contare su tecniche di rappresentazione sparsa. Comunque non ci serve rappresentare la matrice approssimata per LSI, ma soltanto i vettori latenti associati ai documenti e ai termini, che sono densi ma di dimensione ridotta ($k$ invece di $m$ o $n$).

In particolare:
- $D_k = \Sigma_k V_k^T$ rappresenta i documenti nello spazio latente ridotto, ogni colonna è un vettore latente associato a un documento
- $T_k = U_k \Sigma_k$ rappresenta i termini nello spazio latente ridotto, ogni riga è un vettore latente associato a un termine
- La query viene rappresentata nello spazio latente ridotto come $q_k = q^T U_k \Sigma_k^{-1}$, dove $q$ è la rappresentazione originale della query nello spazio dei termini. Questa operazione è detta **folding in**. Perché si fa in questo modo? Intuitivamente perché la query vive nello spazio originale dei termini, $U_k$ è la matrice che proietta i termini nello spazio latente, e $\Sigma_k^{-1}$ serve a scalare correttamente la query in base all'importanza dei concetti latenti.

Si può interpretare una cella $A_k[t,d]$ come prodotto scalare tra vettore termine e vettore documento nello spazio latente. In questo senso è utile sfruttare la rappresentazione simmetrica:
$$A_k = U_k \Sigma_k V_k^T = (U_k \Sigma_k) (V_k^T) = T_k D_k $$
e possiamo quindi dire che:
$$A_k[t,d] = T_k[t,:] \cdot D_k[:,d] = \langle T_k[t,:], D_k[:,d] \rangle$$
Qui sotto la rappresentazione grafica di LSI con due dimensioni latenti $k = 2$, considerando 6 documenti molto semplici divisi in due "topics": animali e macchine. 

<img src="img/lsi.png" alt="LSI" width="400">

Dopodiché per calcolare il ranking via LSI si fa folding in sulla query e si calcola la similarità con i documenti via cosine similarity. La principale caratteristica di LSI è che permette di recuperare documenti semanticamente vicini anche quando non condividono esattamente gli stessi termini.

Si passa all'applicazione di LSI al dataset reale di 20 Newsgroups.

L'utilità di LSI e in particolare SVD è il fatto che, nel caso di una collezione grande, esistono algoritmi iterativi efficienti che permettono di calcolare solo le prime $k$ componenti principali, senza dover calcolare la SVD completa della matrice dei termini-documenti, che sarebbe computazionalmente difficile. 

Si osserva dal grafico seguente come la varianza (dove con varianza si intende la quantità di informazione spiegata) aumenta con il numero di dimensioni latenti $k$. Tipicamente per applicazioni reali il valore di $k$ va da 50 a 300, ma dipende molto dal dataset e dall'applicazione specifica.

<img src="img/varianza.png" alt="Varianza spiegata" width="400">

In particolare:
- con $k$ **troppo piccolo** --> rischiamo di perdere troppa informazione, non catturando la struttura semantica dei dati
- con $k$ **troppo grande** --> ci avviciniamo troppo all'enorme spazio originale, mantenendo più rumore che non permette di generalizzare correttamente

Serve quindi un $k$ intermedio che catturi la struttura semantica principale senza mantenere troppo rumore.

Con LSI come già detto è possibile rappresentare nello spazio latente, oltre che i documenti, anche i termini. L'idea è che, mentre **nello spazio originale due termini sono considerati simili se appaiono negli stessi documenti**, nello spazio latente **sono simili se hanno comportamenti simili rispetto ai principali pattern della collezione**.



## Query exp. e considerazioni ulteriori
LSI e lo spazio latente possono essere sfruttati per fare **query expansion**: se ad esempio la query contiene "space" e nello spazio latente troviamo termini vicini come "universe" o "galaxy", possiamo espandere la query con questi termini per aumentare le possibilità di recuperare documenti rilevanti che non contengono esattamente "space" ma parlano comunque di argomenti simili.

Inoltre **per garantire una buona usability si deve comunque dare fiducia all'utente e ai termini precisi che decide di usare nella query** --> i termini aggiunti **dovrebbero avere un peso minore rispetto a quelli originali nel calcolo dello score**.

Parlando invece di efficienza, usare LSI come l'abbiamo descritto prevede che per il calcolo della similarità si confrontino tutti i documenti k-dimensionali con la query proiettata nello spazio latente, questo è chiaramente proibitivo per collezioni molto grandi.  
Per questo motivo molti sistemi di retrieval usano una **strategia a due fasi**:
1. **Candidate generation**: si usa un indice inverso tradizionale per recuperare un insieme di documenti candidati, ad esempio quelli che contengono almeno uno dei termini della query oppure primi $N$ documenti più rilevanti secondo un modello rapido come BM25
2. **Re-ranking**: si applica LSI solo a questo insieme ristretto di candidati per calcolare la similarità con la query proiettata e ottenere il ranking finale

In questo senso la query expansion può aiutare nella prima fase: si aggiungono termini semanticamente vicini alla query e si permette quindi all'indice inverso di recuperare documenti che magari non ne contengono esattamente i termini originali ma potrebbero essere comunque rilevanti. Si mantiene al contempo la ricerca efficiente dal momento che l'insieme di candidati è ridotto sfruttando le posting list dell'indice inverso.

**Differenze rispetto a TF-IDF e VSM in generale**: mentre TF-IDF confronta query e documenti nello spazio originale dei termini, LSI proietta tutto in uno spazio latente di dimensione ridotta, dove la similarità è basata su concetti latenti piuttosto che sui termini esatti. In questo modo termini diversi possono essere considerati simili se condividono un comportamento simile nei documenti. Il parametro $k$ controlla il compromesso tra compressione e perdita di informazione. 

**NB LSI non è un modello neurale**: LSI non comprende il significato come un modello linguistico moderno, ma usa la struttura globale della matrice term-docs per catturare pattern latenti. Non ha capacità di generalizzazione o di apprendimento come un modello neurale, e inoltre le rappresentazioni dense di termini e documenti non è risultato di un addestramento predittivo su un task (come invece avviene nei modelli neurali tipo BERT o modelli transformer), ma derivano semplicemente dalla decomposizione matriciale di $A$.

Quindi la "semantica" catturata da LSI deriva dalla struttura globale delle co-occorrenze tra termini e documenti nella collezione.

### **Esercizio 2**

<img src="img/esercizio2.png" alt="Esercizio 2" width="400">

1. All'aumentare di $k$, la matrice approssimata $A_k$ si avvicina sempre di più alla matrice originale $A$, quindi l'errore di ricostruzione ($\|A - A_k\|_F$) diminuisce. In particolare, quando $k$ raggiunge il rango di $A$, l'errore diventa zero, poiché $A_k$ coincide con $A$. Quindi, in generale, l'errore di ricostruzione diminuisce al crescere di $k$.
2. Come cambia il plot dei documenti? Con $k = 1$ --> unidimensionale, ogni documento è posizionabile solo su una linea, troppa poca varianza spiegata.  
Con $k = 2$ --> bidimensionale, possiamo vedere una separazione più chiara tra i documenti, plot 2D dove una dimensione racchiude il pattern delineato dal topic della macchine, l'altro degli animali. Con $k = 3$ --> tridimensionale, plot 3D, possiamo vedere ancora più chiaramente la separazione tra i documenti, ma è più difficile da visualizzare.
3. Con $k = 1$ si mantiene solo il pattern più forte della collezione ed è difficile distinguere i documenti in modo chiaro.
4. Se $k$ è max, allora si torna alla matrice term-docs originale --> errore di ricostruzione zero. Tuttavia in questo modo non abbiamo filtraggio del rumore, non si ha più il vantaggio di LSI di catturare la struttura semantica principale. 

